<a href="https://colab.research.google.com/github/njpinton/CMSC178IP/blob/main/Storage%20and%20Compression/Storage_and_Compression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CMSC 178IP: Digital Image Processing
## Supplementary Lab: Image Types, Storage Formats, and Compression Analysis

### 📚 Learning Objectives
In this hands-on lab, we will explore the fundamental concepts covered in our lecture slides:

- **Image Types**: Binary, Grayscale, RGB, and Indexed color representations
- **Storage Requirements**: Mathematical calculations for file sizes and memory usage
- **File Formats**: BMP, PNG, JPEG, GIF characteristics and compression ratios
- **Compression Analysis**: Quality vs file size trade-offs

### 🔧 What We'll Build
- Sample images demonstrating each image type
- Storage requirement calculations with real data
- File format comparisons with actual compression ratios
- Interactive visualizations of compression effects

### 📖 Connection to Lecture
This lab directly implements the theoretical concepts from:
- **Slides 6-10**: Image type representations and characteristics
- **Slides 11-15**: Storage format analysis and comparisons  
- **Slides 16-20**: Image coding and compression fundamentals
- **Slides 21-23**: Real-world applications and format selection

---
Let's start by setting up our environment and creating sample images! 🚀

In [1]:
# ============================================================================
# SETUP: Import Libraries and Configure Environment
# ============================================================================

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
import cv2
import os
import io
from skimage import data, color
from skimage.util import img_as_ubyte
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Configure plotting parameters for better visualization
plt.style.use('default')
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['font.size'] = 10
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Display setup confirmation
print("🎯 CMSC 178IP: Digital Image Processing Lab")
print("=" * 55)
print("✅ Libraries imported successfully")
print("✅ Plotting parameters configured")
print("✅ Environment ready for image analysis")
print("=" * 55)

# Check library versions for reproducibility
print("\n📋 Library Versions:")
print(f"   • NumPy: {np.__version__}")
print(f"   • Matplotlib: {plt.matplotlib.__version__}")
print(f"   • PIL: {Image.__version__}")
print(f"   • OpenCV: {cv2.__version__}")
print(f"   • Pandas: {pd.__version__}")

print("\n🚀 Ready to explore image types and storage formats!")

ModuleNotFoundError: No module named 'cv2'

---

## Section 1: Creating Sample Images 🎨

### 🎯 Objective
Create representative sample images for each major image type covered in our lecture:

1. **RGB Color Image** - Full 24-bit color with gradients and shapes
2. **Binary Image** - 1-bit black and white through thresholding  
3. **Grayscale Image** - 8-bit intensity values
4. **Indexed Color Image** - Limited palette with color lookup table

### 🔬 Technical Approach
We'll programmatically generate a base RGB image containing:
- **Color gradients** to demonstrate RGB channel mixing
- **Geometric shapes** to show edge definition across different image types
- **Varied intensities** to test thresholding and quantization effects

### 📐 Image Specifications
- **Dimensions**: 200×150 pixels (30,000 total pixels)
- **Base Format**: 24-bit RGB color
- **Content**: Gradients + geometric shapes for comprehensive testing

### 🔗 Connection to Slides
This directly implements concepts from:
- **Slide 8**: RGB color model and channel representation
- **Slide 6**: Binary image creation through thresholding
- **Slide 7**: Grayscale conversion methods
- **Slide 10**: Indexed color and palette quantization

Let's create our sample images! ⚡

In [ ]:
# ============================================================================
# SECTION 1: IMAGE CREATION FUNCTIONS
# ============================================================================

def create_base_rgb_image():
    """
    Create a base RGB image with gradients and geometric shapes

    Returns:
        numpy.ndarray: RGB image array (height, width, 3)
    """

    # Define image dimensions
    width, height = 200, 150
    print(f"📐 Creating base RGB image: {width}×{height} pixels")

    # Initialize RGB array
    rgb_image = np.zeros((height, width, 3), dtype=np.uint8)

    # Create color gradients across the image
    print("🎨 Adding color gradients...")
    for i in range(height):
        for j in range(width):
            # Red: increases left to right
            # Green: increases top to bottom
            # Blue: decreases left to right
            rgb_image[i, j] = [
                int(255 * j / width),           # Red gradient (0-255)
                int(255 * i / height),          # Green gradient (0-255)
                int(255 * (1 - j/width))        # Blue gradient (255-0)
            ]

    # Add geometric shapes for edge detection testing
    print("🔺 Adding geometric shapes...")

    # White circle (top-left)
    cv2.circle(rgb_image, (50, 50), 20, (255, 255, 255), -1)

    # Black rectangle (center-right)
    cv2.rectangle(rgb_image, (120, 80), (180, 120), (0, 0, 0), -1)

    # Magenta circle (top-right)
    cv2.circle(rgb_image, (150, 40), 15, (255, 0, 255), -1)

    # Yellow triang

---

## Section 2: Mathematical Analysis & Storage Calculations 🧮

### 🎯 Objective
Calculate exact storage requirements and analyze the mathematical relationships between:
- **Image dimensions** and total pixels
- **Bit depth** and storage requirements  
- **Color channels** and memory usage
- **Theoretical vs actual** file sizes

### 📊 Analysis Framework
For each image type, we'll calculate:

| Property | Formula | Example |
|----------|---------|---------|
| **Total Pixels** | Width × Height | 200 × 150 = 30,000 |
| **Bits Required** | Pixels × Bit Depth | 30,000 × 8 = 240,000 bits |
| **Bytes Required** | Total Bits ÷ 8 | 240,000 ÷ 8 = 30,000 bytes |
| **Memory Ratio** | RGB bytes ÷ Binary bytes | 90,000 ÷ 3,750 = 24:1 |

### 🔢 Key Calculations
- **Binary**: 1 bit/pixel → 30,000 bits = 3,750 bytes
- **Grayscale**: 8 bits/pixel → 240,000 bits = 30,000 bytes  
- **RGB**: 24 bits/pixel → 720,000 bits = 90,000 bytes
- **Indexed**: 8 bits/pixel + palette → ~30,048 bytes

### 🎓 Educational Value
Students will see how the theoretical formulas from slides 4, 6-10 translate into real storage requirements and understand the memory efficiency trade-offs between different image representations.

In [ ]:
# ============================================================================
# SECTION 2: MATHEMATICAL ANALYSIS & STORAGE CALCULATIONS
# ============================================================================

def analyze_image_storage_requirements(binary, grayscale, rgb, indexed_array):
    """
    Comprehensive analysis of storage requirements for each image type

    Args:
        binary, grayscale, rgb, indexed_array: Image arrays

    Returns:
        dict: Complete analysis of storage properties
    """

    height, width = grayscale.shape
    total_pixels = width * height

    print(f"🔍 Analyzing storage for {width}×{height} image ({total_pixels:,} pixels)")

    # Define properties for each image type
    image_properties = {
        'Binary': {
            'description': '1-bit per pixel (black/white only)',
            'dimensions': f"{width}×{height}",
            'total_pixels': total_pixels,
            'channels': 1,
            'bit_depth': 1,
            'bits_per_pixel': 1,
            'color_range': '0-1 (2 possible values)',
            'unique_colors': len(np.unique(binary)),

            # Storage calculations
            'total_bits': total_pixels * 1,
            'theoretical_bytes': (total_pixels * 1) / 8,
            'actual_array_bytes': binary.nbytes,
            'memory_efficiency': 'Most efficient'
        },

        'Grayscale': {
            'description': '8-bit per pixel (256 gray levels)',
            'dimensions': f"{width}×{height}",
            'total_pixels': total_pixels,
            'channels': 1,
            'bit_depth': 8,
            'bits_per_pixel': 8,
            'color_range': '0-255 (256 gray levels)',
            'unique_colors': len(np.unique(grayscale)),

            # Storage calculations
            'total_bits': total_pixels * 8,
            'theoretical_bytes': total_pixels * 1,
            'actual_array_bytes': grayscale.nbytes,
            'memory_efficiency': '8× more than binary'
        },

        'RGB': {
            'description': '24-bit per pixel (16.7M colors)',
            'dimensions': f"{width}×{height}",
            'total_pixels': total_pixels,
            'channels': 3,
            'bit_depth': 8,
            'bits_per_pixel': 24,
            'color_range': '0-255 per channel (16,777,216 colors)',
            'unique_colors': len(np.unique(rgb.reshape(-1, 3), axis=0)),

            # Storage calculations
            'total_bits': total_pixels * 24,
            'theoretical_bytes': total_pixels * 3,
            'actual_array_bytes': rgb.nbytes,
            'memory_efficiency': '24× more than binary'
        },

        'Indexed': {
            'description': '8-bit indices + color palette',
            'dimensions': f"{width}×{height}",
            'total_pixels': total_pixels,
            'channels': 1,
            'bit_depth': 8,
            'bits_per_pixel': 8,
            'color_range': '0-15 indices (16 palette colors)',
            'unique_colors': 16,  # Quantized to 16 colors

            # Storage calculations (image + palette)
            'total_bits': (total_pixels * 8) + (16 * 24),  # pixel indices + palette
            'theoretical_bytes': total_pixels + (16 * 3),   # indices + RGB palette
            'actual_array_bytes': indexed_array.nbytes,
            'memory_efficiency': '8× more than binary + palette overhead'
        }
    }

    return image_properties

def display_storage_analysis(properties):
    """Display comprehensive storage analysis in formatted tables"""

    print("\n📊 DETAILED STORAGE ANALYSIS")
    print("=" * 80)

    # Main properties table
    print(f"{'Type':<10} {'Dimensions':<12} {'Channels':<9} {'Bit Depth':<10} {'BPP':<5} {'Unique Colors':<14} {'Storage (bytes)':<15}")
    print("-" * 80)

    for img_type, props in properties.items():
        print(f"{img_type:<10} {props['dimensions']:<12} {props['channels']:<9} "
              f"{props['bit_depth']:<10} {props['bits_per_pixel']:<5} "
              f"{props['unique_colors']:<14} {props['theoretical_bytes']:<15.0f}")

    print("\n🧮 MATHEMATICAL CALCULATIONS")
    print("=" * 60)

    # Detailed calculations for each type
    for img_type, props in properties.items():
        print(f"\n{img_type.upper()} IMAGE:")
        print(f"   Formula: {props['total_pixels']:,} pixels × {props['bits_per_pixel']} bits/pixel = {props['total_bits']:,} bits")
        print(f"   Storage: {props['total_bits']:,} bits ÷ 8 = {props['theoretical_bytes']:.0f} bytes")
        print(f"   Memory: {props['actual_array_bytes']} bytes (NumPy array)")
        print(f"   Colors: {props['unique_colors']} unique values")
        print(f"   Range: {props['color_range']}")

    # Memory efficiency comparison
    print(f"\n⚖️  MEMORY EFFICIENCY COMPARISON")
    print("=" * 45)

    binary_size = properties['Binary']['theoretical_bytes']

    for img_type, props in properties.items():
        if img_type != 'Binary':
            ratio = props['theoretical_bytes'] / binary_size
            print(f"{img_type:>10} vs Binary: {ratio:>6.1f}× more storage")

    return properties

# Execute storage analysis
print("🚀 SECTION 2: Mathematical Analysis & Storage Calculations")
print("=" * 55)

# Analyze storage requirements
storage_properties = analyze_image_storage_requirements(
    binary_img, gray_img, rgb_img, indexed_array
)

# Display comprehensive analysis
analyzed_props = display_storage_analysis(storage_properties)

# Calculate key insights
total_pixels = storage_properties['Binary']['total_pixels']
binary_bytes = storage_properties['Binary']['theoretical_bytes']
rgb_bytes = storage_properties['RGB']['theoretical_bytes']

print(f"\n💡 KEY INSIGHTS:")
print(f"   • Total pixels analyzed: {total_pixels:,}")
print(f"   • Most efficient: Binary ({binary_bytes:.0f} bytes)")
print(f"   • Least efficient: RGB ({rgb_bytes:.0f} bytes)")
print(f"   • RGB memory overhead: {rgb_bytes/binary_bytes:.0f}× larger than binary")
print(f"   • Color representation trade-off clearly demonstrated!")
print("=" * 55)

---

## Section 3: Visual Comparison & Image Type Demonstration 📊

### 🎯 Objective
Create professional visualizations that clearly demonstrate the differences between image types and their storage characteristics. This section brings the mathematical concepts to life through visual comparison.

### 🖼️ Visualization Strategy
We'll create a comprehensive comparison showing:

1. **Side-by-side image display** - Visual differences between types
2. **Storage requirement bars** - Relative size comparisons  
3. **Color distribution analysis** - Histogram comparisons
4. **Quality assessment** - Visual fidelity trade-offs

### 📈 Educational Benefits
Students will visually understand:
- How **bit depth reduction** affects image quality
- Why **binary images** work well for certain applications
- How **color quantization** impacts visual appearance
- The relationship between **storage efficiency** and **visual quality**

### 🔗 Connection to Slides
This implements visual concepts from:
- **Slide 5**: Image types overview and comparison
- **Slide 6**: Binary image characteristics and applications  
- **Slide 7**: Grayscale representation and bit depth
- **Slide 8-9**: Color models and RGB representation
- **Slide 10**: Indexed color and palette systems

Let's create compelling visualizations! 🎨

In [ ]:
# ============================================================================
# SECTION 3: VISUAL COMPARISON & IMAGE TYPE DEMONSTRATION
# ============================================================================

def create_comprehensive_image_comparison(binary, grayscale, rgb, indexed, properties):
    """
    Create a comprehensive visual comparison of all image types

    Args:
        binary, grayscale, rgb, indexed: Image arrays
        properties: Storage analysis results
    """

    print("🎨 Creating comprehensive image type comparison...")

    # Create main comparison figure
    fig = plt.figure(figsize=(16, 12))
    fig.suptitle('CMSC 178IP: Complete Image Type Analysis', fontsize=18, fontweight='bold', y=0.95)

    # Define grid layout (3 rows, 4 columns)
    gs = fig.add_gridspec(3, 4, height_ratios=[2, 1, 1.2], hspace=0.3, wspace=0.3)

    # Top row: Image comparisons
    images = [
        (binary, 'Binary Image\n1 bit/pixel', 'gray'),
        (grayscale, 'Grayscale Image\n8 bits/pixel', 'gray'),
        (rgb, 'RGB Color Image\n24 bits/pixel', None),
        (indexed, 'Indexed Color\n8 bits/pixel + palette', None)
    ]

    for i, (img, title, cmap) in enumerate(images):
        ax = fig.add_subplot(gs[0, i])
        ax.imshow(img, cmap=cmap)

        # Add detailed title with storage info
        img_type = ['Binary', 'Grayscale', 'RGB', 'Indexed'][i]
        storage_bytes = properties[img_type]['theoretical_bytes']
        unique_colors = properties[img_type]['unique_colors']

        full_title = f"{title}\n{storage_bytes:.0f} bytes\n{unique_colors} colors"
        ax.set_title(full_title, fontsize=11, fontweight='bold')
        ax.axis('off')

        # Add colored border based on efficiency
        colors = ['green', 'blue', 'red', 'orange']
        for spine in ax.spines.values():
            spine.set_edgecolor(colors[i])
            spine.set_linewidth(3)

    # Middle row: Storage comparison bar chart
    ax_storage = fig.add_subplot(gs[1, :])

    types = list(properties.keys())
    storage_values = [properties[t]['theoretical_bytes'] for t in types]
    colors = ['green', 'blue', 'red', 'orange']

    bars = ax_storage.bar(types, storage_values, color=colors, alpha=0.7, edgecolor='black')
    ax_storage.set_title('Storage Requirements Comparison', fontsize=14, fontweight='bold')
    ax_storage.set_ylabel('Storage (bytes)')
    ax_storage.set_xlabel('Image Type')

    # Add value labels on bars
    for bar, value in zip(bars, storage_values):
        height = bar.get_height()
        ax_storage.text(bar.get_x() + bar.get_width()/2., height + max(storage_values)*0.01,
                       f'{value:.0f}B', ha='center', va='bottom', fontweight='bold')

    # Add efficiency ratios
    binary_size = storage_values[0]
    for i, (bar, value) in enumerate(zip(bars, storage_values)):
        if i > 0:  # Skip binary (reference)
            ratio = value / binary_size
            ax_storage.text(bar.get_x() + bar.get_width()/2., height/2,
                           f'{ratio:.1f}×', ha='center', va='center',
                           fontsize=10, fontweight='bold', color='white')

    # Bottom row: Color distribution histograms
    hist_data = [
        (binary.flatten(), 'Binary Distribution', 'green'),
        (grayscale.flatten(), 'Grayscale Distribution', 'blue'),
        (rgb[:,:,0].flatten(), 'RGB Red Channel', 'red'),
        (rgb[:,:,1].flatten(), 'RGB Green Channel', 'green')
    ]

    for i, (data, title, color) in enumerate(hist_data):
        ax = fig.add_subplot(gs[2, i])
        ax.hist(data, bins=50, color=color, alpha=0.7, edgecolor='black')
        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.set_xlabel('Intensity Value')
        ax.set_ylabel('Pixel Count')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print("✅ Comprehensive image comparison created!")

def create_detailed_analysis_summary(properties):
    """Create detailed analysis summary with key insights"""

    print("\n📋 DETAILED VISUAL ANALYSIS SUMMARY")
    print("=" * 60)

    # Calculate key metrics
    total_pixels = properties['Binary']['total_pixels']
    binary_bytes = properties['Binary']['theoretical_bytes']
    rgb_bytes = properties['RGB']['theoretical_bytes']

    print(f"\n🔍 IMAGE SPECIFICATIONS:")
    print(f"   • Dimensions: {properties['Binary']['dimensions']} pixels")
    print(f"   • Total pixels: {total_pixels:,}")
    print(f"   • Test content: Gradients + geometric shapes")

    print(f"\n📊 STORAGE EFFICIENCY ANALYSIS:")
    for img_type, props in properties.items():
        efficiency = (binary_bytes / props['theoretical_bytes']) * 100
        print(f"   • {img_type:<10}: {props['theoretical_bytes']:>7.0f} bytes ({efficiency:>5.1f}% efficiency vs RGB)")

    print(f"\n🎨 COLOR REPRESENTATION ANALYSIS:")
    for img_type, props in properties.items():
        color_density = props['unique_colors'] / total_pixels * 100
        print(f"   • {img_type:<10}: {props['unique_colors']:>6} unique colors ({color_density:.2f}% color diversity)")

    print(f"\n⚖️  PRACTICAL TRADE-OFFS:")
    print(f"   • Binary: Minimal storage, extreme quality loss")
    print(f"   • Grayscale: 8× binary size, good for scientific data")
    print(f"   • RGB: 24× binary size, full color reproduction")
    print(f"   • Indexed: Similar to grayscale, limited but efficient color")

    return properties

# Execute visualization section
print("🚀 SECTION 3: Visual Comparison & Image Type Demonstration")
print("=" * 60)

# Create comprehensive comparison
create_comprehensive_image_comparison(
    binary_img, gray_img, rgb_img, indexed_array, storage_properties
)

# Generate detailed summary
detailed_analysis = create_detailed_analysis_summary(storage_properties)

print("\n💡 KEY VISUAL INSIGHTS:")
print("   ✓ Binary images preserve shape information but lose detail")
print("   ✓ Grayscale maintains detail with 8× storage increase")
print("   ✓ RGB provides full color at 24× storage cost")
print("   ✓ Indexed color balances quality and efficiency")
print("   ✓ Storage requirements scale predictably with bit depth")
print("=" * 60)

---

## Section 4: File Format Analysis & Compression Comparison 📁

### 🎯 Objective
Analyze real-world file formats and their compression characteristics by saving our sample images in different formats and measuring actual file sizes versus theoretical calculations.

### 📊 Formats to Analyze
We'll test major image formats covered in slides 11-15:

| Format | Type | Compression | Best Use Case |
|--------|------|-------------|---------------|
| **BMP** | Uncompressed | None | Maximum quality |
| **PNG** | Lossless | DEFLATE | Web graphics, transparency |
| **JPEG** | Lossy | DCT-based | Photographs |
| **GIF** | Lossless | LZW | Simple graphics, animation |

### 🔬 Analysis Metrics
For each format + image type combination:
- **Actual file size** (bytes saved to disk)
- **Compression ratio** (original ÷ compressed)
- **Storage efficiency** compared to theoretical size
- **Quality preservation** assessment

### 📈 Expected Results
- **BMP**: Largest files, no compression
- **PNG**: Good compression for graphics  
- **JPEG**: Excellent compression for photos
- **GIF**: Efficient for limited-color images

### 🎓 Learning Outcomes
Students will understand:
- How **compression algorithms** perform with different image content
- Why **format selection** depends on image characteristics
- The relationship between **file size** and **image complexity**
- Real-world **storage vs quality** trade-offs

Let's dive into format analysis! 📊

In [ ]:
# ============================================================================
# SECTION 4: FILE FORMAT ANALYSIS & COMPRESSION TESTING
# ============================================================================

def save_images_in_multiple_formats(binary, grayscale, rgb, indexed_pil):
    """
    Save images in different formats and analyze compression characteristics

    Args:
        binary, grayscale, rgb: NumPy image arrays
        indexed_pil: PIL indexed image object

    Returns:
        list: Comprehensive format analysis results
    """

    print("💾 Testing file format compression characteristics...")

    # Convert NumPy arrays to PIL Images for saving
    binary_pil = Image.fromarray(binary, mode='L')
    grayscale_pil = Image.fromarray(grayscale, mode='L')
    rgb_pil = Image.fromarray(rgb, mode='RGB')

    # Define format testing matrix
    format_tests = {
        'BMP': {
            'description': 'Windows Bitmap - Uncompressed',
            'compression': 'None',
            'images': [
                ('Binary', binary_pil),
                ('Grayscale', grayscale_pil),
                ('RGB', rgb_pil)
            ]
        },
        'PNG': {
            'description': 'Portable Network Graphics - Lossless',
            'compression': 'DEFLATE (LZ77 + Huffman)',
            'images': [
                ('Binary', binary_pil),
                ('Grayscale', grayscale_pil),
                ('RGB', rgb_pil),
                ('Indexed', indexed_pil)
            ]
        },
        'JPEG': {
            'description': 'Joint Photographic Experts Group - Lossy',
            'compression': 'DCT + Quantization + Huffman',
            'images': [
                ('Grayscale', grayscale_pil),
                ('RGB', rgb_pil)
            ]  # JPEG doesn't handle binary well
        },
        'GIF': {
            'description': 'Graphics Interchange Format - Indexed',
            'compression': 'LZW (Dictionary-based)',
            'images': [
                ('Indexed', indexed_pil)
            ]  # GIF works best with indexed/palette images
        }
    }

    # Storage for results
    format_results = []

    # Test each format
    for format_name, format_info in format_tests.items():
        print(f"\n🔧 Testing {format_name} format...")
        print(f"   Description: {format_info['description']}")
        print(f"   Compression: {format_info['compression']}")

        for img_name, img_pil in format_info['images']:
            try:
                # Save to memory buffer to measure exact size
                buffer = io.BytesIO()

                # Set format-specific parameters
                save_kwargs = {}
                if format_name == 'JPEG':
                    save_kwargs = {'quality': 85, 'optimize': True}
                elif format_name == 'PNG':
                    save_kwargs = {'optimize': True}
                elif format_name == 'GIF':
                    save_kwargs = {'optimize': True}

                # Save image and measure size
                img_pil.save(buffer, format=format_name, **save_kwargs)
                file_size = buffer.tell()

                # Get theoretical uncompressed size
                if img_name == 'Binary':
                    theoretical_size = storage_properties['Binary']['theoretical_bytes']
                elif img_name == 'Grayscale':
                    theoretical_size = storage_properties['Grayscale']['theoretical_bytes']
                elif img_name == 'RGB':
                    theoretical_size = storage_properties['RGB']['theoretical_bytes']
                else:  # Indexed
                    theoretical_size = storage_properties['Indexed']['theoretical_bytes']

                # Calculate metrics
                compression_ratio = theoretical_size / file_size if file_size > 0 else 0
                size_reduction = ((theoretical_size - file_size) / theoretical_size) * 100 if theoretical_size > 0 else 0

                # Store results
                result = {
                    'Format': format_name,
                    'Image Type': img_name,
                    'File Size (bytes)': file_size,
                    'Theoretical Size (bytes)': theoretical_size,
                    'Compression Ratio': f"{compression_ratio:.2f}:1",
                    'Size Reduction': f"{size_reduction:.1f}%",
                    'Efficiency': 'Excellent' if compression_ratio > 5 else 'Good' if compression_ratio > 2 else 'Fair'
                }

                format_results.append(result)

                print(f"      {img_name}: {file_size} bytes ({compression_ratio:.2f}:1 compression)")

            except Exception as e:
                print(f"      ❌ Could not save {img_name} as {format_name}: {str(e)}")

    print(f"\n✅ Format testing completed! Analyzed {len(format_results)} format combinations.")
    return format_results

def analyze_format_performance(results):
    """Analyze and display format performance characteristics"""

    print("\n📊 COMPREHENSIVE FORMAT ANALYSIS")
    print("=" * 80)

    # Convert to DataFrame for easier analysis
    df = pd.DataFrame(results)

    # Display main results table
    print("\n📋 DETAILED FORMAT COMPARISON:")
    display_columns = ['Format', 'Image Type', 'File Size (bytes)', 'Compression Ratio', 'Size Reduction', 'Efficiency']
    print(df[display_columns].to_string(index=False, max_colwidth=15))

    # Analyze best performers
    print(f"\n🏆 BEST PERFORMING COMBINATIONS:")

    # Convert compression ratios to numeric for analysis
    df['Numeric Compression'] = df['Compression Ratio'].str.replace(':1', '').astype(float)
    best_compression = df.loc[df['Numeric Compression'].idxmax()]

    print(f"   • Best Compression: {best_compression['Format']} with {best_compression['Image Type']}")
    print(f"     Ratio: {best_compression['Compression Ratio']}, Size: {best_compression['File Size (bytes)']} bytes")

    # Analyze format characteristics
    print(f"\n🔍 FORMAT CHARACTERISTICS ANALYSIS:")

    format_summary = {}
    for format_name in df['Format'].unique():
        format_data = df[df['Format'] == format_name]
        avg_compression = format_data['Numeric Compression'].mean()
        image_types = format_data['Image Type'].tolist()

        format_summary[format_name] = {
            'avg_compression': avg_compression,
            'supported_types': image_types,
            'best_type': format_data.loc[format_data['Numeric Compression'].idxmax(), 'Image Type']
        }

        print(f"   • {format_name:>6}: Avg {avg_compression:.2f}:1, Best with {format_summary[format_name]['best_type']}")

    return df, format_summary

# Execute file format analysis
print("🚀 SECTION 4: File Format Analysis & Compression Testing")
print("=" * 65)

# Test all format combinations
compression_results = save_images_in_multiple_formats(
    binary_img, gray_img, rgb_img, indexed_pil
)

# Analyze results
results_df, format_analysis = analyze_format_performance(compression_results)

# Display key insights
print(f"\n💡 KEY COMPRESSION INSIGHTS:")
print(f"   ✓ Uncompressed formats (BMP) show theoretical vs actual size relationships")
print(f"   ✓ Lossless formats (PNG) provide good compression without quality loss")
print(f"   ✓ Lossy formats (JPEG) achieve high compression for photographic content")
print(f"   ✓ Specialized formats (GIF) excel with their target image types")
print(f"   ✓ Compression effectiveness varies significantly by image content")
print("=" * 65)

---

## Section 5: JPEG Quality Analysis & Compression Trade-offs ⚖️

### 🎯 Objective
Demonstrate the **quality vs file size trade-off** that's central to lossy compression by testing JPEG compression at multiple quality levels and analyzing the results both numerically and visually.

### 🔬 Experimental Design
We'll compress our RGB sample image using JPEG quality settings from **10% to 100%** and measure:

- **File size reduction** at each quality level
- **Compression ratio** achieved
- **Visual quality preservation**
- **Compression artifacts** introduction

### 📊 Quality Levels to Test
| Quality | Expected Result | Typical Use Case |
|---------|-----------------|------------------|
| **10-30%** | Heavy compression, visible artifacts | Thumbnails, previews |
| **40-60%** | Balanced compression | Web images, email |
| **70-85%** | Good quality | Standard photography |
| **90-100%** | Minimal compression | Professional/archival |

### 🎓 Learning Outcomes
This analysis demonstrates key concepts from slides 18-20:
- **Lossy compression principles** - How information is strategically discarded
- **Quality vs size trade-offs** - The fundamental compression decision
- **DCT compression artifacts** - Blocking effects at low quality
- **Practical quality selection** - Real-world compression decisions

### 📈 Expected Pattern
We should see:
- **Exponential file size growth** with quality increases
- **Diminishing returns** at very high quality levels
- **Visible artifacts** becoming prominent below 50% quality
- **Sweet spot** around 75-85% for most applications

Let's explore the compression quality spectrum! 📊

In [ ]:
# ============================================================================
# SECTION 5: JPEG QUALITY ANALYSIS & COMPRESSION TRADE-OFFS (FIXED)
# ============================================================================

def get_quality_category(quality):
    """Categorize JPEG quality levels"""
    if quality >= 90:
        return "Excellent"
    elif quality >= 80:
        return "Very Good"
    elif quality >= 70:
        return "Good"
    elif quality >= 50:
        return "Acceptable"
    elif quality >= 30:
        return "Poor"
    else:
        return "Very Poor"

def perform_jpeg_quality_analysis(rgb_image, quality_levels=None):
    """
    Comprehensive JPEG quality vs compression analysis

    Args:
        rgb_image: RGB image array for testing
        quality_levels: List of JPEG quality levels to test

    Returns:
        list: Detailed analysis results for each quality level
    """

    if quality_levels is None:
        quality_levels = [10, 20, 30, 40, 50, 60, 70, 75, 80, 85, 90, 95, 100]

    print(f"📸 Performing JPEG quality analysis across {len(quality_levels)} quality levels...")

    # Convert to PIL for JPEG saving
    rgb_pil = Image.fromarray(rgb_image)
    uncompressed_size = storage_properties['RGB']['theoretical_bytes']

    quality_results = []
    compressed_images = {}  # Store for visual comparison

    print(f"   Uncompressed RGB size: {uncompressed_size:,} bytes")
    print(f"   Testing quality levels: {quality_levels}")

    for quality in quality_levels:
        try:
            # Save to memory buffer with specified quality
            buffer = io.BytesIO()
            rgb_pil.save(buffer, format='JPEG', quality=quality, optimize=True)
            compressed_size = buffer.tell()

            # Load back for visual comparison (simulate real-world usage)
            buffer.seek(0)  # Reset buffer position to beginning
            compressed_pil = Image.open(buffer)
            compressed_array = np.array(compressed_pil.convert('RGB'))  # Ensure RGB mode
            compressed_images[quality] = compressed_array

            # Calculate compression metrics
            compression_ratio = uncompressed_size / compressed_size
            size_reduction = ((uncompressed_size - compressed_size) / uncompressed_size) * 100
            bits_per_pixel = (compressed_size * 8) / (rgb_image.shape[0] * rgb_image.shape[1])

            # Calculate quality metrics (simplified PSNR estimation)
            # Ensure both arrays have same shape and type
            original_float = rgb_image.astype(float)
            compressed_float = compressed_array.astype(float)

            # Handle potential shape differences
            if original_float.shape != compressed_float.shape:
                print(f"      Warning: Shape mismatch at quality {quality}% - using approximation")
                mse = 100  # Use approximation for shape mismatches
            else:
                mse = np.mean((original_float - compressed_float) ** 2)

            psnr = 20 * np.log10(255.0 / np.sqrt(mse)) if mse > 0 else float('inf')

            # Get quality category
            quality_cat = get_quality_category(quality)

            result = {
                'Quality': quality,
                'File Size (bytes)': compressed_size,
                'Compression Ratio': compression_ratio,
                'Size Reduction (%)': size_reduction,
                'Bits per Pixel': bits_per_pixel,
                'PSNR (dB)': psnr,
                'Quality Category': quality_cat
            }

            quality_results.append(result)

            print(f"      Quality {quality:3d}%: {compressed_size:5d} bytes ({compression_ratio:5.1f}:1) - {quality_cat}")

        except Exception as e:
            print(f"      ❌ Error at quality {quality}%: {str(e)}")
            # Continue with other quality levels
            continue

    print(f"✅ JPEG quality analysis completed! Processed {len(quality_results)} quality levels.")
    return quality_results, compressed_images

def visualize_jpeg_quality_analysis(quality_data, compressed_images, original_rgb):
    """Create comprehensive visualization of JPEG quality analysis"""

    print("\n📊 Creating JPEG quality analysis visualizations...")

    if not quality_data:
        print("❌ No quality data available for visualization")
        return None

    # Create comprehensive figure
    fig = plt.figure(figsize=(18, 14))
    fig.suptitle('JPEG Compression Quality Analysis - CMSC 178IP', fontsize=16, fontweight='bold')

    # Convert results to DataFrame
    df = pd.DataFrame(quality_data)

    # Layout: 3 rows, 4 columns
    gs = fig.add_gridspec(3, 4, height_ratios=[1.2, 1.2, 1], hspace=0.3, wspace=0.3)

    # Top row: Quality vs File Size relationship
    ax1 = fig.add_subplot(gs[0, :2])
    ax1.plot(df['Quality'], df['File Size (bytes)'], 'bo-', linewidth=2, markersize=6)
    ax1.set_title('Quality vs File Size Relationship', fontsize=12, fontweight='bold')
    ax1.set_xlabel('JPEG Quality (%)')
    ax1.set_ylabel('File Size (bytes)')
    ax1.grid(True, alpha=0.3)

    # Add annotations for key points
    if len(df) >= 3:
        key_indices = [0, len(df)//2, -1]  # First, middle, last
        for i in key_indices:
            row = df.iloc[i]
            ax1.annotate(f"{row['File Size (bytes)']}B\n{row['Compression Ratio']:.1f}:1",
                        xy=(row['Quality'], row['File Size (bytes)']),
                        xytext=(10, 10), textcoords='offset points',
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7),
                        fontsize=8)

    # Top row: Compression Ratio vs Quality
    ax2 = fig.add_subplot(gs[0, 2:])
    ax2.plot(df['Quality'], df['Compression Ratio'], 'ro-', linewidth=2, markersize=6)
    ax2.set_title('Compression Ratio vs Quality', fontsize=12, fontweight='bold')
    ax2.set_xlabel('JPEG Quality (%)')
    ax2.set_ylabel('Compression Ratio (X:1)')
    ax2.grid(True, alpha=0.3)

    # Middle row: Visual quality comparison (selected quality levels)
    quality_samples = [10, 50, 85, 100]
    available_qualities = list(compressed_images.keys())

    for i, target_quality in enumerate(quality_samples):
        ax = fig.add_subplot(gs[1, i])

        if target_quality == 100:
            # Use original for 100% comparison
            ax.imshow(original_rgb)
            title = f"Original\n(Uncompressed)"
        elif target_quality in available_qualities:
            ax.imshow(compressed_images[target_quality])
            result = df[df['Quality'] == target_quality]
            if not result.empty:
                result = result.iloc[0]
                title = f"Quality {target_quality}%\n{result['File Size (bytes)']} bytes"
            else:
                title = f"Quality {target_quality}%"
        elif available_qualities:
            # Find closest available quality
            closest_quality = min(available_qualities, key=lambda x: abs(x - target_quality))
            ax.imshow(compressed_images[closest_quality])
            result = df[df['Quality'] == closest_quality]
            if not result.empty:
                result = result.iloc[0]
                title = f"Quality {closest_quality}%\n{result['File Size (bytes)']} bytes"
            else:
                title = f"Quality {closest_quality}%"
        else:
            # No compressed images available
            ax.text(0.5, 0.5, 'No Image\nAvailable', ha='center', va='center', transform=ax.transAxes)
            title = f"Quality {target_quality}%\n(Not Available)"

        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.axis('off')

    # Bottom row: Analysis metrics (only if we have data)
    if len(df) > 0:
        ax3 = fig.add_subplot(gs[2, :2])
        ax3.plot(df['Quality'], df['PSNR (dB)'], 'go-', linewidth=2, markersize=6)
        ax3.set_title('Peak Signal-to-Noise Ratio vs Quality', fontsize=12, fontweight='bold')
        ax3.set_xlabel('JPEG Quality (%)')
        ax3.set_ylabel('PSNR (dB)')
        ax3.grid(True, alpha=0.3)

        # Bottom row: Bits per pixel
        ax4 = fig.add_subplot(gs[2, 2:])
        ax4.plot(df['Quality'], df['Bits per Pixel'], 'mo-', linewidth=2, markersize=6)
        ax4.set_title('Bits per Pixel vs Quality', fontsize=12, fontweight='bold')
        ax4.set_xlabel('JPEG Quality (%)')
        ax4.set_ylabel('Bits per Pixel')
        ax4.grid(True, alpha=0.3)

        # Add reference line at 24 bpp (uncompressed RGB)
        ax4.axhline(y=24, color='red', linestyle='--', alpha=0.7, label='Uncompressed RGB (24 bpp)')
        ax4.legend()

    plt.tight_layout()
    plt.show()

    # Display numerical analysis
    if len(df) > 0:
        print("\n📋 DETAILED JPEG QUALITY ANALYSIS:")
        print("-" * 90)
        print(f"{'Quality':<8} {'Size (B)':<10} {'Ratio':<8} {'Reduction':<10} {'BPP':<6} {'PSNR':<8} {'Category':<12}")
        print("-" * 90)

        for _, row in df.iterrows():
            print(f"{row['Quality']:<8} {row['File Size (bytes)']:<10} "
                  f"{row['Compression Ratio']:<8.1f} {row['Size Reduction (%)']:<10.1f} "
                  f"{row['Bits per Pixel']:<6.2f} {row['PSNR (dB)']:<8.1f} {row['Quality Category']:<12}")

    return df

# Execute JPEG quality analysis
print("🚀 SECTION 5: JPEG Quality Analysis & Compression Trade-offs")
print("=" * 65)

# Perform comprehensive JPEG analysis
try:
    jpeg_results, jpeg_images = perform_jpeg_quality_analysis(rgb_img)

    if jpeg_results:
        # Create visualizations and analysis
        jpeg_df = visualize_jpeg_quality_analysis(jpeg_results, jpeg_images, rgb_img)

        # Generate insights
        if jpeg_df is not None and not jpeg_df.empty:
            min_size = jpeg_df['File Size (bytes)'].min()
            max_size = jpeg_df['File Size (bytes)'].max()
            sweet_spot = jpeg_df[(jpeg_df['Quality'] >= 75) & (jpeg_df['Quality'] <= 85)]

            print(f"\n💡 KEY JPEG COMPRESSION INSIGHTS:")
            print(f"   📊 Size range: {min_size:,} to {max_size:,} bytes ({max_size/min_size:.1f}× variation)")

            if not sweet_spot.empty:
                print(f"   🎯 Sweet spot (75-85% quality): {sweet_spot['File Size (bytes)'].mean():.0f} bytes average")

            quality_85 = jpeg_df[jpeg_df['Quality']==85]
            if not quality_85.empty:
                print(f"   ⚖️  Quality 85%: {quality_85.iloc[0]['Compression Ratio']:.1f}:1 compression ratio")

            print(f"   📉 Below 50% quality: Significant visual artifacts appear")
            print(f"   📈 Above 90% quality: Diminishing returns for file size")
            print(f"   ✅ Successfully analyzed {len(jpeg_df)} quality levels")
        else:
            print("   ❌ No valid JPEG results to analyze")
    else:
        print("   ❌ No JPEG compression results obtained")

except Exception as e:
    print(f"❌ Error in JPEG analysis: {str(e)}")
    print("   This may be due to PIL/JPEG library configuration issues")
    import traceback
    traceback.print_exc()

print("=" * 65)

---

## Section 6: Practical Format Selection & Real-World Applications 🌍

### 🎯 Objective
Synthesize all our analysis into **practical guidelines** for image format selection based on real-world scenarios and application requirements.

### 🏢 Industry Applications
We'll analyze format selection for:

| Application | Primary Concern | Recommended Formats | Rationale |
|-------------|----------------|-------------------|-----------|
| **Web Development** | Loading Speed + Quality | WebP → JPEG → PNG | Progressive enhancement |
| **Medical Imaging** | Data Integrity | DICOM, PNG, TIFF | Zero data loss required |
| **E-commerce** | User Experience | Progressive JPEG + WebP | Fast loading, good quality |
| **Social Media** | Storage + Bandwidth | Aggressive JPEG | Volume optimization |
| **Professional Photography** | Archive Quality | RAW + TIFF/PNG | Maximum preservation |
| **Mobile Apps** | Battery + Storage | HEIF, WebP, optimized JPEG | Efficiency focused |

### 📊 Decision Framework
Our analysis provides a systematic approach to format selection:

1. **Quality Requirements** - Lossless vs acceptable quality loss
2. **Storage Constraints** - Available space and bandwidth
3. **Compatibility Needs** - Target platform support
4. **Processing Resources** - Encoding/decoding speed requirements
5. **Future-proofing** - Format longevity and evolution

### 🎓 Educational Summary
This section consolidates learning from all previous sections:
- **Image type characteristics** (Sections 1-3)
- **Storage calculations** (Section 2)  
- **Format compression analysis** (Section 4)
- **Quality vs size trade-offs** (Section 5)

Let's create practical selection guidelines! 🚀

In [ ]:
# ============================================================================
# SECTION 6: COMPREHENSIVE SUMMARY & PRACTICAL RECOMMENDATIONS
# ============================================================================

def create_comprehensive_summary_analysis():
    """
    Generate comprehensive summary of all analysis performed
    """

    print("📊 COMPREHENSIVE IMAGE ANALYSIS SUMMARY")
    print("=" * 60)

    # Image specifications summary
    height, width = rgb_img.shape[:2]
    total_pixels = height * width

    print(f"\n🖼️  IMAGE SPECIFICATIONS:")
    print(f"   • Dimensions: {width}×{height} = {total_pixels:,} pixels")
    print(f"   • Content: Gradients + geometric shapes for comprehensive testing")
    print(f"   • Created: Programmatically for educational consistency")

    # Storage analysis summary
    print(f"\n💾 STORAGE ANALYSIS RESULTS:")
    storage_summary = []

    for img_type, props in storage_properties.items():
        efficiency_vs_rgb = (storage_properties['RGB']['theoretical_bytes'] / props['theoretical_bytes'])
        storage_summary.append({
            'Type': img_type,
            'Storage (bytes)': props['theoretical_bytes'],
            'Efficiency vs RGB': f"{efficiency_vs_rgb:.1f}×",
            'Unique Colors': props['unique_colors'],
            'Best Use': get_best_use_case(img_type)
        })

    storage_df = pd.DataFrame(storage_summary)
    print(storage_df.to_string(index=False))

    # Format analysis summary
    if 'compression_results' in globals():
        print(f"\n📁 FORMAT COMPRESSION SUMMARY:")
        format_df = pd.DataFrame(compression_results)

        # Group by format and calculate averages
        format_summary = format_df.groupby('Format').agg({
            'File Size (bytes)': 'mean',
            'Image Type': 'count'
        }).round(0)
        format_summary.columns = ['Avg File Size (bytes)', 'Types Tested']

        print(format_summary.to_string())

    # JPEG analysis summary
    if 'jpeg_results' in globals() and jpeg_results:
        print(f"\n📸 JPEG QUALITY ANALYSIS SUMMARY:")
        jpeg_summary_df = pd.DataFrame(jpeg_results)

        # Key quality levels analysis
        key_qualities = [10, 50, 85, 100] if len(jpeg_summary_df) > 0 else []

        print(f"{'Quality':<8} {'Size (bytes)':<12} {'Compression':<12} {'Category':<12}")
        print("-" * 50)

        for quality in key_qualities:
            quality_row = jpeg_summary_df[jpeg_summary_df['Quality'] == quality]
            if not quality_row.empty:
                row = quality_row.iloc[0]
                print(f"{quality:<8} {row['File Size (bytes)']:<12} "
                      f"{row['Compression Ratio']:<12.1f} {row['Quality Category']:<12}")

    return storage_summary

def get_best_use_case(img_type):
    """Return best use case for each image type"""
    use_cases = {
        'Binary': 'OCR, barcodes, line art',
        'Grayscale': 'Medical, scientific imaging',
        'RGB': 'Photography, web graphics',
        'Indexed': 'Simple graphics, animations'
    }
    return use_cases.get(img_type, 'General purpose')

def create_practical_selection_guide():
    """
    Create practical format selection guide based on our analysis
    """

    print(f"\n🎯 PRACTICAL FORMAT SELECTION GUIDE")
    print("=" * 50)

    # Define selection scenarios based on our analysis
    scenarios = {
        'Web Development': {
            'priority': 'Loading speed + Visual quality',
            'primary': 'JPEG (85% quality)',
            'fallback': 'PNG for graphics with transparency',
            'modern': 'WebP with JPEG fallback',
            'reasoning': f'Our analysis shows 85% JPEG provides {jpeg_df.loc[jpeg_df["Quality"]==85, "Compression Ratio"].iloc[0]:.1f}:1 compression' if 'jpeg_df' in globals() and not jpeg_df.empty and 85 in jpeg_df['Quality'].values else 'Good compression with acceptable quality'
        },
        'Medical/Scientific': {
            'priority': 'Data integrity',
            'primary': 'PNG or TIFF (lossless)',
            'fallback': 'DICOM for medical standards',
            'modern': 'JPEG-LS for lossless compression',
            'reasoning': f'Lossless formats preserve all {total_pixels:,} pixels of diagnostic data'
        },
        'E-commerce': {
            'priority': 'User experience',
            'primary': 'Progressive JPEG (75-85%)',
            'fallback': 'Multiple resolutions',
            'modern': 'WebP with lazy loading',
            'reasoning': 'Balance between quality and loading speed for product images'
        },
        'Social Media': {
            'priority': 'Storage efficiency',
            'primary': 'Aggressive JPEG (60-75%)',
            'fallback': 'Automatic compression',
            'modern': 'HEIF for mobile uploads',
            'reasoning': f'Volume optimization - millions of images need efficient storage'
        },
        'Professional Photography': {
            'priority': 'Maximum quality',
            'primary': 'RAW + TIFF for archival',
            'fallback': 'PNG for web delivery',
            'modern': 'JPEG XL for future-proofing',
            'reasoning': f'Preserve full {storage_properties["RGB"]["theoretical_bytes"]:,} bytes of color data'
        },
        'Mobile Applications': {
            'priority': 'Battery + Storage',
            'primary': 'Optimized JPEG',
            'fallback': 'PNG for UI elements',
            'modern': 'HEIF, WebP, AVIF',
            'reasoning': 'Hardware-accelerated decoding and storage efficiency'
        }
    }

    for scenario, details in scenarios.items():
        print(f"\n📱 {scenario.upper()}:")
        print(f"   Priority: {details['priority']}")
        print(f"   Primary: {details['primary']}")
        print(f"   Fallback: {details['fallback']}")
        print(f"   Modern: {details['modern']}")
        print(f"   Reasoning: {details['reasoning']}")

    return scenarios

def create_decision_flowchart_data():
    """
    Create data for decision flowchart visualization
    """

    print(f"\n🌳 FORMAT SELECTION DECISION TREE")
    print("=" * 40)

    decision_tree = """
    START: What type of image content?
    │
    ├── Text/Line Art/Simple Graphics
    │   ├── Few colors? → GIF or PNG-8
    │   └── Many colors? → PNG-24
    │
    ├── Photographs/Complex Images
    │   ├── Quality critical? → PNG or TIFF
    │   ├── Web delivery? → JPEG (85% quality)
    │   └── Mobile app? → WebP or HEIF
    │
    ├── Medical/Scientific Data
    │   └── Always → PNG, TIFF, or DICOM (lossless)
    │
    └── Animation Required?
        ├── Simple animation → GIF
        └── Complex animation → WebP or video formats
    """

    print(decision_tree)

    # Create summary recommendations table
    recommendations = {
        'Use Case': ['Web Graphics', 'Photography', 'Medical Data', 'Mobile Apps', 'Print Media', 'Archival'],
        'Primary Format': ['PNG', 'JPEG', 'PNG/TIFF', 'WebP/HEIF', 'TIFF', 'TIFF/PNG'],
        'Quality Setting': ['Lossless', '80-90%', 'Lossless', '75-85%', 'Lossless', 'Lossless'],
        'Rationale': [
            'Transparency + lossless',
            'Good compression + quality',
            'Data integrity required',
            'Efficiency + compatibility',
            'Maximum quality',
            'Long-term preservation'
        ]
    }

    rec_df = pd.DataFrame(recommendations)
    print(f"\n📋 QUICK REFERENCE TABLE:")
    print(rec_df.to_string(index=False))

    return decision_tree, recommendations

# Execute comprehensive summary
print("🚀 SECTION 6: Practical Format Selection & Real-World Applications")
print("=" * 70)

# Generate comprehensive analysis summary
summary_data = create_comprehensive_summary_analysis()

# Create practical selection guide
selection_scenarios = create_practical_selection_guide()

# Create decision flowchart data
decision_data, quick_ref = create_decision_flowchart_data()

print(f"\n💡 KEY INSIGHTS FROM COMPLETE ANALYSIS:")
print(f"   ✅ Image type selection impacts storage by up to 24× factor")
print(f"   ✅ Format selection can achieve 2× to 50× compression ratios")
print(f"   ✅ Quality settings provide fine-grained size vs quality control")
print(f"   ✅ Application requirements determine optimal format choice")
print(f"   ✅ Modern formats offer better efficiency but require compatibility planning")
print("=" * 70)

In [ ]:
# ============================================================================
# FINAL SECTION: COMPLETE ANALYSIS VISUALIZATION & COURSE CONNECTION
# ============================================================================

def create_master_visualization_summary():
    """
    Create comprehensive final visualization summarizing all analysis
    """

    print("🎨 Creating master summary visualization...")

    # Create large comprehensive figure
    fig = plt.figure(figsize=(20, 16))
    fig.suptitle('CMSC 178IP: Complete Image Analysis - From Theory to Practice',
                 fontsize=18, fontweight='bold', y=0.96)

    # Define complex grid layout
    gs = fig.add_gridspec(4, 6, height_ratios=[1, 1, 1, 0.8], hspace=0.3, wspace=0.3)

    # Row 1: Image type comparison
    image_types = [
        (binary_img, 'Binary\n1 bpp', 'gray'),
        (gray_img, 'Grayscale\n8 bpp', 'gray'),
        (rgb_img, 'RGB\n24 bpp', None)
    ]

    for i, (img, title, cmap) in enumerate(image_types):
        ax = fig.add_subplot(gs[0, i*2:(i*2)+2])
        ax.imshow(img, cmap=cmap)
        ax.set_title(f'{title}\n{storage_properties[["Binary", "Grayscale", "RGB"][i]]["theoretical_bytes"]:.0f} bytes',
                    fontsize=12, fontweight='bold')
        ax.axis('off')

    # Row 2: Storage comparison and format analysis
    ax_storage = fig.add_subplot(gs[1, :3])
    types = list(storage_properties.keys())
    storage_vals = [storage_properties[t]['theoretical_bytes'] for t in types]
    colors = ['green', 'blue', 'red', 'orange']

    bars = ax_storage.bar(types, storage_vals, color=colors, alpha=0.7)
    ax_storage.set_title('Storage Requirements by Image Type', fontsize=12, fontweight='bold')
    ax_storage.set_ylabel('Storage (bytes)')

    # Add efficiency labels
    binary_size = storage_vals[0]
    for bar, val in zip(bars, storage_vals):
        efficiency = val / binary_size
        height = bar.get_height()
        ax_storage.text(bar.get_x() + bar.get_width()/2, height/2,
                       f'{efficiency:.0f}×' if efficiency > 1 else '1×',
                       ha='center', va='center', fontweight='bold', color='white')

    # Format compression comparison (if available)
    if 'compression_results' in globals():
        ax_formats = fig.add_subplot(gs[1, 3:])

        # Process format data for visualization
        format_df = pd.DataFrame(compression_results)
        format_summary = format_df.groupby('Format')['File Size (bytes)'].mean().sort_values()

        bars_fmt = ax_formats.bar(format_summary.index, format_summary.values,
                                 color=['red', 'blue', 'green', 'orange'][:len(format_summary)])
        ax_formats.set_title('Average File Size by Format', fontsize=12, fontweight='bold')
        ax_formats.set_ylabel('Average File Size (bytes)')
        ax_formats.tick_params(axis='x', rotation=45)

    # Row 3: JPEG quality analysis (if available)
    if 'jpeg_df' in globals() and not jpeg_df.empty:
        ax_jpeg1 = fig.add_subplot(gs[2, :3])
        ax_jpeg1.plot(jpeg_df['Quality'], jpeg_df['File Size (bytes)'], 'bo-', linewidth=2)
        ax_jpeg1.set_title('JPEG Quality vs File Size', fontsize=12, fontweight='bold')
        ax_jpeg1.set_xlabel('JPEG Quality (%)')
        ax_jpeg1.set_ylabel('File Size (bytes)')
        ax_jpeg1.grid(True, alpha=0.3)

        ax_jpeg2 = fig.add_subplot(gs[2, 3:])
        ax_jpeg2.plot(jpeg_df['Quality'], jpeg_df['Compression Ratio'], 'ro-', linewidth=2)
        ax_jpeg2.set_title('JPEG Quality vs Compression Ratio', fontsize=12, fontweight='bold')
        ax_jpeg2.set_xlabel('JPEG Quality (%)')
        ax_jpeg2.set_ylabel('Compression Ratio (X:1)')
        ax_jpeg2.grid(True, alpha=0.3)

    # Row 4: Course connection and key insights
    ax_text = fig.add_subplot(gs[3, :])
    ax_text.axis('off')

    # Create course connection text
    course_text = f"""
COURSE CONNECTION - CMSC 178IP Digital Image Processing:

SLIDES 6-10 (Image Types): Demonstrated {len(storage_properties)} image types with {total_pixels:,} pixels each
- Binary images: {storage_properties['Binary']['unique_colors']} unique values, {storage_properties['Binary']['theoretical_bytes']:.0f} bytes storage
- Grayscale images: {storage_properties['Grayscale']['unique_colors']} unique values, 8× binary storage
- RGB images: {storage_properties['RGB']['unique_colors']} unique colors, 24× binary storage

SLIDES 11-15 (Storage Formats): Analyzed {len(compression_results) if 'compression_results' in globals() else 'multiple'} format combinations
- BMP: Uncompressed baseline for comparison
- PNG: Lossless compression with transparency support
- JPEG: Lossy compression optimized for photographs
- GIF: LZW compression ideal for simple graphics

SLIDES 16-20 (Compression): {'JPEG quality analysis across ' + str(len(jpeg_df)) + ' quality levels' if 'jpeg_df' in globals() and not jpeg_df.empty else 'Compression trade-off analysis'}
- Quality vs Size trade-offs clearly demonstrated
- Compression ratios from 2:1 to 50:1 achieved
- Visual quality preservation vs storage efficiency

PRACTICAL APPLICATIONS: Format selection depends on application requirements, storage constraints, and quality needs
    """

    ax_text.text(0.02, 0.95, course_text, transform=ax_text.transAxes, fontsize=10,
                verticalalignment='top', fontfamily='monospace',
                bbox=dict(boxstyle='round,pad=0.5', facecolor='lightblue', alpha=0.8))

    plt.tight_layout()
    plt.show()

    print("✅ Master visualization created successfully!")

def generate_course_learning_summary():
    """
    Generate final learning summary connecting to course objectives
    """

    print(f"\n🎓 COURSE LEARNING OUTCOMES ACHIEVED")
    print("=" * 50)

    learning_outcomes = {
        "Image Types Understanding": {
            "achieved": "✅ MASTERED",
            "evidence": f"Created and analyzed {len(storage_properties)} image types with quantitative storage analysis",
            "slide_connection": "Slides 6-10: Binary, Grayscale, RGB, Indexed color systems"
        },
        "Storage Format Knowledge": {
            "achieved": "✅ MASTERED",
            "evidence": f"Tested {len(compression_results) if 'compression_results' in globals() else 'multiple'} format combinations with real compression ratios",
            "slide_connection": "Slides 11-15: BMP, PNG, JPEG, GIF characteristics and applications"
        },
        "Compression Principles": {
            "achieved": "✅ MASTERED",
            "evidence": f"Analyzed {'JPEG quality across ' + str(len(jpeg_df)) + ' levels' if 'jpeg_df' in globals() and not jpeg_df.empty else 'compression trade-offs'} with quantitative metrics",
            "slide_connection": "Slides 16-20: Lossless vs lossy compression, quality trade-offs"
        },
        "Practical Application": {
            "achieved": "✅ MASTERED",
            "evidence": f"Developed decision framework and format selection guide for 6 real-world scenarios",
            "slide_connection": "Slides 21-23: Advanced algorithms, applications, summary"
        }
    }

    for outcome, details in learning_outcomes.items():
        print(f"\n📚 {outcome.upper()}:")
        print(f"   Status: {details['achieved']}")
        print(f"   Evidence: {details['evidence']}")
        print(f"   Connection: {details['slide_connection']}")

    # Final statistics summary
    print(f"\n📊 FINAL ANALYSIS STATISTICS:")
    print(f"   • Total pixels analyzed: {total_pixels:,}")
    print(f"   • Image types tested: {len(storage_properties)}")
    print(f"   • Storage calculations: {len(storage_properties)} theoretical + actual measurements")
    print(f"   • Format combinations: {len(compression_results) if 'compression_results' in globals() else 'Multiple'} tested")
    print(f"   • JPEG quality levels: {len(jpeg_df) if 'jpeg_df' in globals() and not jpeg_df.empty else 'Multiple'} analyzed")
    print(f"   • Practical scenarios: 6 real-world applications covered")

    return learning_outcomes

def create_next_steps_guide():
    """
    Provide guidance for next steps in digital image processing
    """

    print(f"\n🚀 NEXT STEPS IN DIGITAL IMAGE PROCESSING")
    print("=" * 50)

    next_topics = {
        "Week ? - Image Enhancement": {
            "topics": "Spatial filtering, histogram processing, contrast enhancement",
            "build_on": "Understanding of image representation from this week",
            "preparation": "Review grayscale analysis and pixel value distributions"
        },
        "Week ? - Frequency Domain": {
            "topics": "Fourier transforms, frequency filtering, DCT analysis",
            "build_on": "JPEG compression concepts introduced today",
            "preparation": "Understand how spatial patterns relate to frequency content"
        },
        "Week ? - Image Restoration": {
            "topics": "Noise reduction, deblurring, inverse filtering",
            "build_on": "Quality metrics and image degradation understanding",
            "preparation": "Consider how compression artifacts relate to image quality"
        },
        "Advanced Topics": {
            "topics": "Feature detection, segmentation, machine learning applications",
            "build_on": "Solid foundation in image representation and processing",
            "preparation": "Master the fundamentals covered in today's analysis"
        }
    }

    for topic, details in next_topics.items():
        print(f"\n📖 {topic.upper()}:")
        print(f"   Topics: {details['topics']}")
        print(f"   Builds on: {details['build_on']}")
        print(f"   Preparation: {details['preparation']}")

    print(f"\n🎯 RECOMMENDED PRACTICE:")
    print(f"   • Experiment with different image types using this notebook")
    print(f"   • Try the format selection guide with your own images")
    print(f"   • Explore how compression affects different image content")
    print(f"   • Consider the trade-offs in real applications you use daily")

    return next_topics

# Execute final summary and course connection
print("🚀 FINAL SECTION: Complete Analysis Summary & Course Connection")
print("=" * 75)

# Create master visualization
create_master_visualization_summary()

# Generate learning outcomes summary
learning_summary = generate_course_learning_summary()

# Create next steps guide
next_steps = create_next_steps_guide()

# Final completion message
print(f"\n" + "="*75)
print(f"🎉 LABORATORY ANALYSIS COMPLETE!")
print(f"📚 All learning objectives achieved with quantitative evidence")
print(f"🔗 Direct connection to lecture slides 6-23 demonstrated")
print(f"💡 Practical knowledge gained for real-world applications")
print(f"🚀 Ready for next week's image enhancement techniques!")
print(f"="*75)

# Save summary for reference
summary_stats = {
    'total_pixels': total_pixels,
    'image_types_analyzed': len(storage_properties),
    'formats_tested': len(compression_results) if 'compression_results' in globals() else 0,
    'jpeg_qualities_tested': len(jpeg_df) if 'jpeg_df' in globals() and not jpeg_df.empty else 0,
    'scenarios_covered': 6
}

print(f"\n💾 Analysis Summary Saved:")
print(f"   Notebook demonstrates complete understanding of CMSC 178IP fundamentals")
print(f"   Ready for practical application and advanced topics")